# YOLO Part B: Classification Training

## Dataset: Plant Disease Classification (Cabai/Chili Pepper)
- **Model:** YOLOv8 Nano Classification
- **Total Images:** 434
- **Training:** 305 images
- **Validation:** 85 images
- **Test:** 44 images
- **Classes:** Multiple disease/health conditions

This notebook trains a YOLO classification model to identify plant health conditions.

## Step 1: Import Libraries and Setup

In [3]:
from ultralytics import YOLO
import os
import sys
from pathlib import Path

# Create output directory
os.makedirs('models', exist_ok=True)

print("=" * 60)
print("PART B: CLASSIFICATION MODEL TRAINING")
print("=" * 60)
print("\n📊 Dataset: Plant Disease Classification (Cabai)")
print("📦 Model: YOLOv8 Nano Classification")
print("💾 Output: models/classification_model.pt")
print("\n" + "=" * 60 + "\n")

PART B: CLASSIFICATION MODEL TRAINING

📊 Dataset: Plant Disease Classification (Cabai)
📦 Model: YOLOv8 Nano Classification
💾 Output: models/classification_model.pt




In [4]:

# Organize classification data into class folders
import shutil
import pandas as pd
from pathlib import Path

print("[0/4] Organizing classification data...")

base_path = Path('.')
splits = ['train', 'valid', 'test']

def organize_by_primary_class(split_name):
    """Organize images into subdirectories by primary class"""
    split = base_path / split_name
    csv_path = split / '_classes.csv'
    
    if not csv_path.exists():
        print(f"No CSV found at {csv_path}")
        return
    
    # Read the CSV
    try:
        df = pd.read_csv(csv_path, skipinitialspace=True)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return
    
    classes = ['healthy', 'curl', 'leaf', 'spot', 'whitefly', 'yellowish']
    organized_count = 0
    
    for idx, row in df.iterrows():
        filename = row['filename'].strip() if isinstance(row['filename'], str) else str(row['filename'])
        
        # Try images subdirectory first, then direct directory  
        src_path = split / 'images' / filename if (split / 'images' / filename).exists() else split / filename
        
        if not src_path.exists():
            continue
        
        # Determine primary class
        primary_class = None
        for cls in classes:
            if cls in df.columns and row[cls] == 1:
                primary_class = cls
                break
        
        if primary_class is None:
            primary_class = 'unknown'
        
        # Create class directory
        class_dir = split / primary_class
        class_dir.mkdir(exist_ok=True)
        
        # Copy image
        dst_path = class_dir / filename
        if not dst_path.exists():
            shutil.copy(str(src_path), str(dst_path))
            organized_count += 1

    print(f"✅ {split_name.capitalize()} data organized ({organized_count} images moved)")

for split in splits:
    organize_by_primary_class(split)

print("✅ Data organization complete!")
print()


[0/4] Organizing classification data...
✅ Train data organized (0 images moved)
✅ Valid data organized (0 images moved)
✅ Test data organized (0 images moved)
✅ Data organization complete!



## Step 2: Load Pretrained Classification Model

In [5]:
print("[1/4] Loading YOLOv8 Nano Classification pretrained model...")
model = YOLO('yolov8n-cls.pt')  # nano classification model
print("✅ Model loaded successfully!")
print(f"Model size: {model.model}")

[1/4] Loading YOLOv8 Nano Classification pretrained model...
✅ Model loaded successfully!
Model size: ClassificationModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-

## Step 3: Train the Classification Model

In [7]:
print("\n[2/4] Starting training...")
print("⏱️  Training plant disease classification model")
print("-" * 60)

results = model.train(
    data='.',                                 # Current directory (classification data structure)
    epochs=5,                                 # Number of training epochs (reduced for faster training)
    imgsz=224,                                # Standard classification image size
    batch=16,                                 # Batch size (reduced for CPU)
    patience=3,                               # Early stopping patience
    device='cpu',                             # CPU device
    save=True,                                # Save model after training
    verbose=True,                             # Verbose output
    workers=4,                                # Number of workers
    project='runs/classify',                  # Project folder
    name='plant_disease_classification',      # Run name
    exist_ok=True                             # Overwrite existing
)

print("-" * 60)
print("✅ Training completed!")


[2/4] Starting training...
⏱️  Training plant disease classification model
------------------------------------------------------------
Ultralytics 8.4.37  Python-3.11.9 torch-2.5.1+cu121 CPU (Intel Core i7-14700HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=., degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

## Step 4: Save the Model

In [8]:
print("\n[3/4] Saving trained model...")
model.save('models/classification_model.pt')
print("✅ Model saved to: models/classification_model.pt")
print(f"File size: {os.path.getsize('models/classification_model.pt') / 1024 / 1024:.2f} MB")


[3/4] Saving trained model...
✅ Model saved to: models/classification_model.pt
File size: 2.84 MB


## Step 5: Test Classification on Sample Image

In [10]:
print("\n[4/4] Testing inference on sample image...")
test_images_path = '../Part_B_Classification/test'

if os.path.exists(test_images_path):
    test_images = [f for f in os.listdir(test_images_path) if f.endswith(('.jpg', '.png'))]
    if test_images:
        sample_image = os.path.join(test_images_path, test_images[0])
        print(f"Testing with: {test_images[0]}")
        
        results = model.predict(
            source=sample_image,
            save=False,
            conf=0.5
        )
        
        # Display prediction
        for result in results:
            print(f"\n✅ Prediction Results:")
            print(f"   Top 1: {result.names[int(result.probs.top1)]} (confidence: {float(result.probs.top1conf):.2%})")
            if hasattr(result.probs, 'top5'):
                print(f"   Top 5 classes:")
                for idx in result.probs.top5:
                    print(f"      - {result.names[int(idx)]}: {float(result.probs.data[int(idx)]):.2%}")
    else:
        print("⚠️  No test images found")
else:
    print("⚠️  Test images folder not found")


[4/4] Testing inference on sample image...
Testing with: Cabai-sehat004_jpg.rf.8bee9af1253a467ef51f60a022949e84.jpg

image 1/1 c:\Users\LOQ 15IRX9\Downloads\DL_Lab_Assignments\YOLO\Part_B_Classification\..\Part_B_Classification\test\Cabai-sehat004_jpg.rf.8bee9af1253a467ef51f60a022949e84.jpg: 224x224 healthy 0.95, curl 0.04, yellowish 0.01, leaf 0.00, whitefly 0.00, 18.6ms
Speed: 4.4ms preprocess, 18.6ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

✅ Prediction Results:
   Top 1: healthy (confidence: 94.55%)
   Top 5 classes:
      - healthy: 94.55%
      - curl: 3.54%
      - yellowish: 1.48%
      - leaf: 0.30%
      - whitefly: 0.14%


## Step 6: Summary

In [12]:
print("\n" + "=" * 60)
print("✅ CLASSIFICATION MODEL TRAINING COMPLETED")
print("=" * 60)
print("\n📁 Output Files:")
print(f"   - Model: models/classification_model.pt")
print(f"   - Runs: runs/classify/plant_disease_classification/")
print(f"\n🎯 Metrics saved in: runs/classify/plant_disease_classification/results.csv")
print("\n" + "=" * 60 + "\n")


✅ CLASSIFICATION MODEL TRAINING COMPLETED

📁 Output Files:
   - Model: models/classification_model.pt
   - Runs: runs/classify/plant_disease_classification/

🎯 Metrics saved in: runs/classify/plant_disease_classification/results.csv


